In [70]:
import pandas as pd
import numpy as np
import os
import sys
sys.path.append(os.path.abspath(".."))
from src.build_dataset import build_dataset
from src.config import TRADING_DAYS
import scipy.stats as stats


In [194]:
def prices_wide(stocks: pd.DataFrame, value_col: str = "adjclose") -> pd.DataFrame:
    """Return a date × ticker table with values from value_col. """
    
    if value_col not in stocks.columns:
        raise ValueError(f"Can not find {value_col} in df")
        
    return stocks.pivot_table(value_col, "date", "ticker")
    

def log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Return daily log returns ln(P_t / P_(t-1)) for each column; the first row is dropped."""   
    
    df = prices.copy()
    
    for col in df.columns:
        df[f"{col}_prev"] = df[col].shift(1)
        df[f"{col}"] = np.log(df[col] / df[f"{col}_prev"])
        df = df.drop([f"{col}_prev"],axis="columns")

    return df.iloc[1:]


def simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Return daily simple_returns _P_t / P_(t-1)) -1 for each column; the first row is dropped."""   
    
    df = prices.copy()
    
    for col in df.columns:
        df[f"{col}_prev"] = df[col].shift(1)
        df[f"{col}"] = (df[col] / df[f"{col}_prev"])-1
        df = df.drop([f"{col}_prev"],axis="columns")

    return df.iloc[1:]


def bank_index(simple_ret: pd.DataFrame) -> pd.Series:
    """ Equal-weighted daily average of bank simple returns, ignoring missing values."""
    return simple_ret.mean(axis = 1).dropna().rename("Bank index")


def describe_returns(returns: pd.DataFrame, periods_per_year: int = TRADING_DAYS) -> pd.DataFrame:
    """Summary statistics of returns for each column, with annualised volatility."""
    return pd.DataFrame({
        "n_days": returns.count(),
        "mean": returns.mean(),
        "std": returns.std(),
        "std_annual": returns.std() * np.sqrt(periods_per_year),
        "skew": returns.skew(),
        "excess_kurtosis": returns.kurt(),
        "min": returns.min(),
        "max": returns.max(),
    })


def normality_test(returns: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """Jarque-Bera normality test for each column."""
    bank = []
    jb_stat = []
    p_value = []
    reject_normality = []

    for col in returns.columns:
        jbt = stats.jarque_bera(returns[col].dropna())
        bank.append(col)
        jb_stat.append(jbt.statistic)
        p_value.append(jbt.pvalue)
        reject_normality.append(jbt.pvalue < alpha)

    return pd.DataFrame(
        {"jb_stat": jb_stat, "p_value": p_value, "reject_normality": reject_normality},
        index=bank,
    )


def rolling_volatility(
    returns: pd.DataFrame, window: int = 21, periods_per_year: int = TRADING_DAYS
) -> pd.DataFrame:
    """Annualised rolling volatility for each column; the first window - 1 rows are NaN."""
    return returns.rolling(window).std() * np.sqrt(periods_per_year)

In [196]:
rolling_volatility(lr)

ticker,Alior,ING,Millennium,PKO BP,Pekao,mBank
date,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-09,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2026-11-02,0.377732,0.340651,0.482970,0.341773,0.324530,0.496063
2026-11-03,0.354368,0.300116,0.442941,0.327047,0.323359,0.467769
2026-11-04,0.347327,0.400749,0.443542,0.332189,0.325836,0.462874


In [66]:
describe_returns(lr)

,n_days,mean,std,std_annual,skew,excess_kurtosis,min,max
ticker,,,,,,,,
Alior,2958,0.000362,0.025483,0.404527,-0.280862,6.834024,-0.221643,0.164658
ING,2962,0.000539,0.019620,0.311461,0.137360,3.954093,-0.116339,0.124162
Millennium,2962,0.000340,0.026660,0.423213,-0.347812,6.903975,-0.240397,0.166729
PKO BP,2962,0.000514,0.020393,0.323730,-0.316262,5.613795,-0.184174,0.124590
Pekao,2962,0.000382,0.020659,0.327954,-0.488095,9.469412,-0.218040,0.158164
mBank,2930,0.000394,0.025465,0.404242,0.256950,5.381664,-0.178323,0.215841


In [ ]:
lr = log_returns(x)
lr.head(3)

In [50]:
lr.mean()

ticker
Alior         0.000362
ING           0.000539
Millennium    0.000340
PKO BP        0.000514
Pekao         0.000382
mBank         0.000394
dtype: float64

In [24]:
sr.head(3)

ticker,Alior,ING,Millennium,PKO BP,Pekao,mBank
date,,,,,,
2015-01-02,-0.011926,-0.010365,-0.018072,-0.000839,-0.003358,0.000000
2015-01-05,-0.021025,-0.024919,-0.029448,-0.018472,-0.023582,-0.032329
2015-01-07,0.011534,0.007407,0.005057,0.003707,0.008626,0.027599


In [18]:
stocks_df, fx_df = build_dataset()
x = prices_wide(stocks_df)
x.head(3)

ticker,Alior,ING,Millennium,PKO BP,Pekao,mBank
date,,,,,,
2015-01-01,51.369934,95.885124,8.30,24.994366,84.049118,473.809082
2015-01-02,50.757290,94.891304,8.15,24.973396,83.766907,473.809082
2015-01-05,49.690102,92.526733,7.91,24.512096,81.791504,458.491150


In [19]:
sr = simple_returns(x)
sr.head(3)

ticker,Alior,ING,Millennium,PKO BP,Pekao,mBank
date,,,,,,
2015-01-02,-0.011926,-0.010365,-0.018072,-0.000839,-0.003358,0.000000
2015-01-05,-0.021025,-0.024919,-0.029448,-0.018472,-0.023582,-0.032329
2015-01-07,0.011534,0.007407,0.005057,0.003707,0.008626,0.027599


In [20]:
lr = log_returns(x)
lr.head(3)

ticker,Alior,ING,Millennium,PKO BP,Pekao,mBank
date,,,,,,
2015-01-02,-0.011998,-0.010419,-0.018238,-0.000839,-0.003363,0.000000
2015-01-05,-0.021249,-0.025234,-0.029890,-0.018644,-0.023865,-0.032863
2015-01-07,0.011468,0.007380,0.005044,0.003700,0.008589,0.027225
